# OpenRouter Free — validação com fallback controlado no notebook

Esta versão não envia o parâmetro `models` ao OpenRouter.

Em vez disso, o próprio notebook:

1. tenta o primeiro modelo gratuito;
2. perante `429`, `404`, `502`, `503` ou outro erro, tenta o modelo seguinte;
3. regista o modelo efetivamente utilizado;
4. gera e valida 2 exemplos N1;
5. guarda e descarrega o JSON.

No Colab, cria o Secret:

`OPENROUTER_API_KEY`

In [1]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

NUMBER_OF_EXAMPLES = 2

OUTPUT_DIR = "/content/openrouter_outputs"
OUTPUT_FILE = (
    f"{OUTPUT_DIR}/n1_openrouter_test_"
    f"{NUMBER_OF_EXAMPLES}_examples.json"
)

MODEL_FALLBACKS = [
    "qwen/qwen3-next-80b-a3b-instruct:free",
    "google/gemma-4-26b-a4b-it:free",
    "nousresearch/hermes-3-llama-3.1-405b:free",
    "nvidia/nemotron-3-super-120b-a12b:free",
    "openai/gpt-oss-20b:free",
    "meta-llama/llama-3.2-3b-instruct:free",
]

TEMPERATURE = 0.8
MAX_TOKENS = 4000

DOMAIN = "Bibliotecas"
FACT_TYPE = "Capacidade"
ENTITY_TYPE = "Biblioteca"
CONTEXT_LENGTH = "curto, com 3 a 5 frases"
FACT_POSITION = "no fim do contexto"
QUESTION_TYPE = "pergunta direta"
DISTRACTOR_RULE = (
    "não incluir informação enganadora nem distratores relevantes"
)

print("Modelos candidatos:")
for model in MODEL_FALLBACKS:
    print("-", model)

Modelos candidatos:
- qwen/qwen3-next-80b-a3b-instruct:free
- google/gemma-4-26b-a4b-it:free
- nousresearch/hermes-3-llama-3.1-405b:free
- nvidia/nemotron-3-super-120b-a12b:free
- openai/gpt-oss-20b:free
- meta-llama/llama-3.2-3b-instruct:free


In [2]:
# ============================================================
# IMPORTS E AUTENTICAÇÃO
# ============================================================

import json
import re
import time
import requests
from pathlib import Path
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError(
        "Adiciona OPENROUTER_API_KEY aos Secrets do Colab."
    )

BASE_URL = "https://openrouter.ai/api/v1"
MODELS_URL = f"{BASE_URL}/models"
CHAT_URL = f"{BASE_URL}/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://colab.research.google.com",
    "X-Title": "Cognitive Curriculum Provider Validation",
}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Configuração pronta.")

Configuração pronta.


In [3]:
# ============================================================
# CONFIRMAR MODELOS GRATUITOS DISPONÍVEIS
# ============================================================

response = requests.get(
    MODELS_URL,
    headers=HEADERS,
    timeout=60,
)

response.raise_for_status()

models = response.json().get("data", [])

available_ids = {
    model.get("id")
    for model in models
    if model.get("id")
}

available_fallbacks = [
    model
    for model in MODEL_FALLBACKS
    if model in available_ids
]

print("Modelos candidatos atualmente disponíveis:")
for model in available_fallbacks:
    print("-", model)

if not available_fallbacks:
    raise RuntimeError(
        "Nenhum modelo candidato está atualmente disponível."
    )

Modelos candidatos atualmente disponíveis:
- qwen/qwen3-next-80b-a3b-instruct:free
- google/gemma-4-26b-a4b-it:free
- nousresearch/hermes-3-llama-3.1-405b:free
- nvidia/nemotron-3-super-120b-a12b:free
- openai/gpt-oss-20b:free
- meta-llama/llama-3.2-3b-instruct:free


## Função de fallback

A função abaixo tenta cada modelo separadamente. Assim conseguimos ver claramente:

- o código HTTP;
- a mensagem de erro;
- o tempo de espera indicado;
- o modelo que acabou por responder.

In [4]:
# ============================================================
# FUNÇÃO DE CHAMADA COM FALLBACK CLIENT-SIDE
# ============================================================

RETRYABLE_CODES = {408, 409, 429, 500, 502, 503, 504}


def extract_retry_after(response, default=12):
    header_value = response.headers.get("Retry-After")

    if header_value:
        try:
            return max(1, int(float(header_value)))
        except ValueError:
            pass

    try:
        metadata = (
            response.json()
            .get("error", {})
            .get("metadata", {})
        )

        value = metadata.get("retry_after_seconds")

        if value is not None:
            return max(1, int(float(value)))
    except Exception:
        pass

    return default


def call_openrouter_with_fallback(
    messages,
    temperature,
    max_tokens,
    require_json=False,
    attempts_per_model=2,
):
    errors = []

    for model_name in available_fallbacks:
        print("\n" + "=" * 90)
        print("A testar:", model_name)
        print("=" * 90)

        for attempt in range(1, attempts_per_model + 1):
            request_payload = {
                "model": model_name,
                "messages": messages,
                "temperature": temperature,
                "max_tokens": max_tokens,
            }

            if require_json:
                request_payload["response_format"] = {
                    "type": "json_object"
                }

            response = requests.post(
                CHAT_URL,
                headers=HEADERS,
                json=request_payload,
                timeout=180,
            )

            if response.ok:
                data = response.json()

                return {
                    "requested_model": model_name,
                    "actual_model": data.get("model", model_name),
                    "data": data,
                }

            print(
                f"Erro HTTP {response.status_code} "
                f"(tentativa {attempt}/{attempts_per_model})"
            )
            print(response.text[:1000])

            errors.append({
                "model": model_name,
                "attempt": attempt,
                "status_code": response.status_code,
                "body": response.text[:2000],
            })

            if (
                response.status_code in RETRYABLE_CODES
                and attempt < attempts_per_model
            ):
                wait_seconds = extract_retry_after(response)

                print(
                    f"Aguardar {wait_seconds} segundos "
                    "antes de repetir o mesmo modelo..."
                )

                time.sleep(wait_seconds + 1)
                continue

            print("Passar ao modelo seguinte.")
            break

    raise RuntimeError(
        "Nenhum modelo respondeu com sucesso.\n\n"
        + json.dumps(errors, ensure_ascii=False, indent=2)
    )

## Teste simples

In [5]:
# ============================================================
# TESTE "OLÁ"
# ============================================================

test_result = call_openrouter_with_fallback(
    messages=[
        {
            "role": "user",
            "content": "Responde apenas com a palavra: Olá",
        }
    ],
    temperature=0,
    max_tokens=20,
    require_json=False,
)

test_data = test_result["data"]

print("\nModelo pedido com sucesso:", test_result["requested_model"])
print("Modelo reportado pela API:", test_result["actual_model"])
print(
    "Resposta:",
    test_data["choices"][0]["message"]["content"],
)


A testar: qwen/qwen3-next-80b-a3b-instruct:free
Erro HTTP 429 (tentativa 1/2)
{"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"qwen/qwen3-next-80b-a3b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations","provider_name":"Venice","is_byok":false,"retry_after_seconds":28,"retry_after_seconds_raw":27.932,"headers":{"Retry-After":"28"}}},"user_id":"user_3GUGkoi4EvP8212cKTVWZHgM89M"}
Aguardar 28 segundos antes de repetir o mesmo modelo...
Erro HTTP 429 (tentativa 2/2)
{"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"qwen/qwen3-next-80b-a3b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations","provider_name":"Venice","is_byok":false,"retry_after_seconds":29,"retry_after_seconds_raw":28.697,"headers":{"Retr

## Prompt N1 de teste

In [6]:
# ============================================================
# PROMPT FIXA
# ============================================================

prompt = f"""
Gera exatamente {NUMBER_OF_EXAMPLES} exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito
num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: {DOMAIN}
- tipo de facto pedido: {FACT_TYPE}
- tipo de entidade principal: {ENTITY_TYPE}
- contexto: {CONTEXT_LENGTH}
- posição do facto que responde à pergunta: {FACT_POSITION}
- tipo de pergunta: {QUESTION_TYPE}
- distratores: {DISTRACTOR_RULE}

REGRAS
1. Cada exemplo deve conter exatamente context, question e answer.
2. A resposta deve aparecer literalmente no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Usa português europeu.
9. Devolve apenas JSON válido.
10. Não uses Markdown.
11. Não acrescentes explicações.

FORMATO OBRIGATÓRIO
{{
  "examples": [
    {{
      "context": "...",
      "question": "...",
      "answer": "..."
    }}
  ]
}}
""".strip()

print(prompt)

Gera exatamente 2 exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito
num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: Bibliotecas
- tipo de facto pedido: Capacidade
- tipo de entidade principal: Biblioteca
- contexto: curto, com 3 a 5 frases
- posição do facto que responde à pergunta: no fim do contexto
- tipo de pergunta: pergunta direta
- distratores: não incluir informação enganadora nem distratores relevantes

REGRAS
1. Cada exemplo deve conter exatamente context, question e answer.
2. A resposta deve aparecer literalmente no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Usa português europeu.
9. Devolve apenas JSON válido.
10. Não uses Markdown.
11. Não acrescentes explicações.

FORMATO OBRIGATÓRI

## Gerar exemplos

In [7]:
# ============================================================
# GERAÇÃO COM FALLBACK
# ============================================================

generation_result = call_openrouter_with_fallback(
    messages=[
        {
            "role": "system",
            "content": (
                "És um gerador de dados estruturados. "
                "Devolve apenas JSON válido."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ],
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    require_json=True,
)

generation_data = generation_result["data"]

requested_model = generation_result["requested_model"]
actual_model = generation_result["actual_model"]

raw_text = (
    generation_data["choices"][0]["message"]["content"]
    .strip()
)

print("\nModelo pedido com sucesso:", requested_model)
print("Modelo reportado pela API:", actual_model)
print("\nResposta:")
print(raw_text[:2000])


A testar: qwen/qwen3-next-80b-a3b-instruct:free
Erro HTTP 429 (tentativa 1/2)
{"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"qwen/qwen3-next-80b-a3b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations","provider_name":"Venice","is_byok":false,"retry_after_seconds":12,"retry_after_seconds_raw":11.828,"headers":{"Retry-After":"12"}}},"user_id":"user_3GUGkoi4EvP8212cKTVWZHgM89M"}
Aguardar 12 segundos antes de repetir o mesmo modelo...
Erro HTTP 429 (tentativa 2/2)
{"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"qwen/qwen3-next-80b-a3b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations","provider_name":"Venice","is_byok":false,"retry_after_seconds":29,"retry_after_seconds_raw":28.613,"headers":{"Retr

## Validar JSON

In [8]:
# ============================================================
# VALIDAÇÃO DO JSON
# ============================================================

def clean_json_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text[len("```json"):]

    if text.startswith("```"):
        text = text[len("```"):]

    if text.endswith("```"):
        text = text[:-3]

    return text.strip()


clean_text = clean_json_text(raw_text)

try:
    payload = json.loads(clean_text)
except json.JSONDecodeError as exc:
    print(clean_text)

    raise ValueError(
        f"O modelo não devolveu JSON válido: {exc}"
    ) from exc

if not isinstance(payload, dict):
    raise TypeError(
        "O resultado principal deve ser um objeto JSON."
    )

if "examples" not in payload:
    raise ValueError(
        "O objeto JSON não contém a chave 'examples'."
    )

examples = payload["examples"]

if not isinstance(examples, list):
    raise TypeError(
        "'examples' deve ser um array JSON."
    )

if len(examples) != NUMBER_OF_EXAMPLES:
    raise ValueError(
        f"Esperava {NUMBER_OF_EXAMPLES} exemplos, "
        f"mas recebi {len(examples)}."
    )

required_fields = {"context", "question", "answer"}

for index, example in enumerate(examples, start=1):
    if not isinstance(example, dict):
        raise TypeError(
            f"O exemplo {index} não é um objeto JSON."
        )

    missing = required_fields - set(example.keys())

    if missing:
        raise ValueError(
            f"Exemplo {index}: campos em falta: "
            f"{sorted(missing)}"
        )

    for field in required_fields:
        if (
            not isinstance(example[field], str)
            or not example[field].strip()
        ):
            raise ValueError(
                f"Exemplo {index}: campo '{field}' inválido."
            )

print(f"JSON válido: {len(examples)} exemplos.")

JSON válido: 2 exemplos.


## Verificar respostas no contexto

In [9]:
# ============================================================
# VERIFICAÇÃO BÁSICA
# ============================================================

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .,:;!?")


all_valid = True

for index, example in enumerate(examples, start=1):
    answer_in_context = (
        normalize(example["answer"])
        in normalize(example["context"])
    )

    print(
        f"Exemplo {index}: "
        f"resposta no contexto = {answer_in_context}"
    )

    if not answer_in_context:
        all_valid = False

if all_valid:
    print(
        "\nTodas as respostas aparecem literalmente "
        "nos contextos."
    )
else:
    print(
        "\nExistem exemplos que necessitam de revisão."
    )

Exemplo 1: resposta no contexto = True
Exemplo 2: resposta no contexto = True

Todas as respostas aparecem literalmente nos contextos.


## Mostrar exemplos

In [10]:
# ============================================================
# APRESENTAÇÃO
# ============================================================

for index, example in enumerate(examples, start=1):
    print("\n" + "=" * 90)
    print(f"EXEMPLO {index}")
    print("=" * 90)

    print("CONTEXTO:")
    print(example["context"])

    print("\nPERGUNTA:")
    print(example["question"])

    print("\nRESPOSTA:")
    print(example["answer"])


EXEMPLO 1
CONTEXTO:
A Biblioteca Municipal de Alvorada inaugurou um novo edifício no centro da cidade. O espaço conta com várias salas de leitura e zonas de estudo individual. A estrutura foi desenhada para acolher o maior número possível de utilizadores. A capacidade máxima da biblioteca é de 150 leitores.

PERGUNTA:
Qual é a capacidade máxima da biblioteca?

RESPOSTA:
150 leitores

EXEMPLO 2
CONTEXTO:
O Centro de Documentação de Arvoredo possui um vasto arquivo histórico. Os investigadores utilizam frequentemente as mesas de consulta para estudar os manuscritos. O acesso é gratuito para todos os residentes da região. Este centro tem capacidade para 40 pessoas em simultâneo.

PERGUNTA:
Qual é a capacidade do centro de documentação?

RESPOSTA:
40 pessoas em simultâneo


## Guardar resultado

In [11]:
# ============================================================
# GUARDAR JSON
# ============================================================

result = {
    "provider": "openrouter",
    "fallback_strategy": "client_side_sequential",
    "candidate_models": available_fallbacks,
    "requested_model_used": requested_model,
    "actual_model": actual_model,
    "number_of_examples": len(examples),
    "generation_settings": {
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "domain": DOMAIN,
        "fact_type": FACT_TYPE,
        "entity_type": ENTITY_TYPE,
        "context_length": CONTEXT_LENGTH,
        "fact_position": FACT_POSITION,
        "question_type": QUESTION_TYPE,
    },
    "usage": generation_data.get("usage", {}),
    "examples": examples,
}

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        result,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Guardado em:")
print(OUTPUT_FILE)

Guardado em:
/content/openrouter_outputs/n1_openrouter_test_2_examples.json


## Descarregar

In [12]:
# ============================================================
# DOWNLOAD
# ============================================================

from google.colab import files

files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>